In [28]:
import pandas as pd
import os

In [29]:
import re
import pandas as pd
import torch
import torch.nn as nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments


In [30]:
file_path = "data/가전/" 
df= pd.DataFrame()
file_list = os.listdir(file_path)

for file in file_list:
    data = pd.read_json(file_path + file)
    df =  pd.concat([df, data], axis=0)

df.reset_index(drop=True, inplace=True)

In [31]:
df['GeneralPolarity'].value_counts()

GeneralPolarity
 1.0    2220
 0.0     944
-1.0     514
Name: count, dtype: int64

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4056 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 380.4+ KB


In [33]:
def normalize_korean_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = re.sub(r"[^0-9a-zA-Z가-힣ㄱ-ㅎㅏ-ㅣ .,!?\"'’‘…~\-]", " ", text)  # 특수문자 제거
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [34]:
df.dropna(subset='GeneralPolarity', inplace=True)

In [35]:
df["RawText"] = df["RawText"].apply(normalize_korean_text)

In [36]:
df = df[df["RawText"].str.len() > 1]  # 1자 이하 제거

In [37]:
df.drop_duplicates('RawText', inplace=True)

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3678 entries, 0 to 4055
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            3678 non-null   int64  
 1   RawText          3678 non-null   object 
 2   Source           3678 non-null   object 
 3   Domain           3678 non-null   object 
 4   MainCategory     3678 non-null   object 
 5   ProductName      3678 non-null   object 
 6   ReviewScore      3678 non-null   int64  
 7   Syllable         3678 non-null   int64  
 8   Word             3678 non-null   int64  
 9   RDate            3678 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          3678 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 373.5+ KB


In [39]:
df = df.rename(columns={
    'GeneralPolarity' : 'label'
})

In [40]:
df['label'] = df['label'].astype(int)

In [41]:
df['label'] = df['label'].map({
    -1: 0, 
    0 : 1, 
    1 : 2
})

In [42]:
df['label'].value_counts()

label
2    2220
1     944
0     514
Name: count, dtype: int64

In [43]:
df = df[ ['RawText', 'label'] ]

In [44]:

# train/test 분리
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

# Hugging Face의 Dataset 객체로 변환 
# BERT 모델에서 바로 사용이 가능한 객체의 형태
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))
train_ds

Dataset({
    features: ['RawText', 'label'],
    num_rows: 2942
})

In [45]:

# ============================================================
# 3. KoBERT 모델 + Tokenizer
# ============================================================

# MODEL_NAME = "skt/kobert-base-v1"
MODEL_NAME = "monologg/koelectra-base-v3-discriminator"
# AutoTokenizer를 이용해 KoBERT 전용 토크나이저 불러오기
# use_fast=False : KoBERT는 sentencepiece 기반이라 fast 버전(WordPiece용)이 아닌 slow 버전을 사용해야 함
# use_fast=True → “빠른(FAST) 토크나이저”
# use_fast=False → “기존(파이썬 기반) 토크나이저”
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

def tok_fn(batch):
    """
    각 데이터(batch)의 'document' 컬럼(텍스트)을 KoBERT 토크나이저로 변환하는 함수.

    truncation=True  → 문장이 모델 최대 입력 길이를 초과하면 자동으로 자름
    max_length=128   → 최대 토큰 길이를 128로 고정 (BERT 입력 시 일반적인 설정)
    return 값에는 input_ids, attention_mask, token_type_ids 등이 포함됨
    """
    return tokenizer(batch["RawText"], truncation=True, max_length=128)
# ----------------------------------------------------------------------
# (2) 학습 데이터셋(train_ds) 토크나이징
# ----------------------------------------------------------------------
# Dataset.map() :
#   - 데이터셋의 각 샘플에 tok_fn 함수를 적용해 토큰화된 결과를 추가
#   - batched=True → 여러 샘플을 한 번에 처리 (속도 향상)
#   - remove_columns=["id", "document"] → 원본 텍스트 컬럼 제거 (모델에는 필요 없음)
train_tok = train_ds.map(tok_fn, batched=False, remove_columns=["RawText"])
test_tok  = test_ds.map(tok_fn,  batched=False, remove_columns=["RawText"])


Map: 100%|██████████| 736/736 [00:00<00:00, 1683.83 examples/s]


In [46]:
train_tok['label']

Column([2, 2, 1, 1, 2])

In [47]:

# ============================================================
# 🧠 BERT 분류 모델 (BERT + Linear Head)
# ============================================================
class BertClsHead(nn.Module):
    def __init__(self, model_name, num_labels=2, dropout=0.1):
        # nn.Module을 상속받은 사용자 정의 모델 초기화
        super().__init__()

        # ------------------------------------------------------------
        # (1) 사전학습된 BERT 모델 로드 (백본)
        # ------------------------------------------------------------
        # BertModel: 사전학습된 BERT Encoder (Transformer 층)
        # from_pretrained(model_name): 지정한 이름의 모델 가중치와 설정을 로드
        self.backbone = BertModel.from_pretrained(model_name)

        # BERT의 히든 벡터 차원 수 (예: 768)
        hidden = self.backbone.config.hidden_size

        # ------------------------------------------------------------
        # (2) Dropout과 Linear Classifier Head 정의
        # ------------------------------------------------------------
        # Dropout: 과적합 방지를 위해 일부 뉴런을 랜덤으로 0으로 만드는 정규화 기법
        self.dropout = nn.Dropout(dropout)

        # Linear Layer: BERT의 출력(768차원)을 num_labels(분류 클래스 수)로 변환
        self.classifier = nn.Linear(hidden, num_labels)

        # BERT의 PAD 토큰 ID를 명시적으로 설정 (마스크 연산 시 안정성 확보)
        self.backbone.config.pad_token_id = tokenizer.pad_token_id


    # ------------------------------------------------------------
    # (3) 순전파(forward) 정의
    # ------------------------------------------------------------
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        """
        input_ids: 토큰화된 입력 문장 (Tensor)
        attention_mask: 실제 단어=1 / 패딩=0 으로 구분하는 마스크
        labels: 학습 시 정답 라벨 (없으면 추론 모드)
        """

        # ① BERT 백본에 입력 전달
        # out.last_hidden_state : 각 토큰의 벡터 (batch_size, seq_len, hidden_size)
        # out.pooler_output : [CLS] 토큰을 별도로 가공한 벡터 (여기서는 직접 CLS 사용)
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)

        # ② [CLS] 토큰 벡터 추출
        # BERT 입력의 첫 번째 토큰([CLS])은 문장 전체를 대표하는 의미로 학습됨
        # 문장 벡터 데이터를 의미
        pooled = out.last_hidden_state[:, 0]  # 첫 번째 토큰([CLS]) 위치의 벡터 선택

        # ③ Dropout + Linear Layer를 통과시켜 분류 로짓(logits) 계산
        logits = self.classifier(self.dropout(pooled))

        # 결과 딕셔너리 초기화
        result = {"logits": logits}

        # ④ 학습 단계: 정답 레이블이 있으면 손실(loss) 계산
        if labels is not None:
            # CrossEntropyLoss: 다중분류용 손실함수 (logits vs 정답 비교)
            loss = nn.CrossEntropyLoss()(logits, labels)
            result["loss"] = loss

        # ⑤ 추론 단계에서는 logits만 반환
        return result

class BertClsHead2(nn.Module):
    def __init__(self, model_name, num_labels=3, dropout=0.3, tokenizer=None):
        super().__init__()
        self.backbone = BertModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, num_labels)

        # ✅ tokenizer를 외부에서 넘겨받아 일관성 보장
        if tokenizer is not None:
            # pad/unk 안전 세팅
            if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
                tokenizer.pad_token = tokenizer.eos_token  # 필요시 대체
            if tokenizer.pad_token_id is not None:
                self.backbone.config.pad_token_id = tokenizer.pad_token_id

            # ✅ 토큰을 추가했거나, 길이가 달라졌다면 임베딩 리사이즈
            if self.backbone.get_input_embeddings().num_embeddings != len(tokenizer):
                self.backbone.resize_token_embeddings(len(tokenizer))

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, labels=None, **kwargs):
        # ✅ input_ids는 반드시 torch.long
        if input_ids is not None and input_ids.dtype != torch.long:
            input_ids = input_ids.long()

        out = self.backbone(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)

        pooled = out.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(pooled))
        result = {"logits": logits}

        if labels is not None:
            # labels도 long + 0..num_labels-1 범위여야 함
            if labels.dtype != torch.long:
                labels = labels.long()
            result["loss"] = nn.CrossEntropyLoss()(logits, labels)
        return result





# ------------------------------------------------------------
# (4) 모델 인스턴스 생성
# ------------------------------------------------------------
# MODEL_NAME: 사전학습된 KoBERT 이름 (예: "skt/kobert-base-v1")
# num_labels=2 : 긍정/부정 2클래스 분류
model = BertClsHead(MODEL_NAME, num_labels=3, dropout=0.3)


You are using a model of type electra to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
Some weights of BertModel were not initialized from the model checkpoint at monologg/koelectra-base-v3-discriminator and are newly initialized: ['embeddings.LayerNorm.bias', 'embeddings.LayerNorm.weight', 'embeddings.position_embeddings.weight', 'embeddings.token_type_embeddings.weight', 'embeddings.word_embeddings.weight', 'encoder.layer.0.attention.output.LayerNorm.bias', 'encoder.layer.0.attention.output.LayerNorm.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.attention.self.key.bias', 'encoder.layer.0.attention.self.key.weight', 'encoder.layer.0.attention.self.query.bias', 'encoder.layer.0.attention.self.query.weight', 'encoder.layer.0.attention.self.value.bias', 'encoder.layer.0.attention.self.value.weight', 'encoder.layer.0.intermediate.dense.bias', 'encode

In [48]:
# ============================================================
# 5. 평가 함수
# ============================================================
def metrics(eval_pred):
    logits, y = eval_pred
    pred = logits.argmax(-1)
    return {"accuracy": accuracy_score(y, pred), "f1": f1_score(y, pred, average="macro")}


In [49]:
# ============================================================
# 6. 학습 설정
# ============================================================

# TrainingArguments : Trainer가 학습할 때 사용할 각종 설정값을 정의하는 객체
args = TrainingArguments(
    # --------------------------------------------------------
    # (1) 출력 디렉토리 설정
    # --------------------------------------------------------
    output_dir="./kobert_from_bertmodel",   # 학습 결과(모델, 로그 등)를 저장할 경로

    # --------------------------------------------------------
    # (2) 배치 크기 설정
    # --------------------------------------------------------
    per_device_train_batch_size=8,         # GPU/CPU 하나당 학습 시 배치 크기
    per_device_eval_batch_size=8,          # 평가 시 배치 크기

    # --------------------------------------------------------
    # (3) 평가 및 저장 주기 설정
    # --------------------------------------------------------
    eval_strategy="epoch",                  # 한 epoch마다 평가 수행
    save_strategy="epoch",                  # 한 epoch마다 모델 저장

    # --------------------------------------------------------
    # (4) 학습 관련 하이퍼파라미터
    # --------------------------------------------------------
    num_train_epochs=5,                     # 학습 epoch 수 (전체 데이터 반복 횟수)
    learning_rate=3e-5,                     # AdamW 옵티마이저의 학습률
    weight_decay=0.01,                      # L2 정규화(가중치 감쇠) 계수
    warmup_ratio=0.1,                       # 학습 초기에 LR을 천천히 올리는 비율 (10%)
    logging_steps=50,                       # 로그를 출력할 step 간격

    # --------------------------------------------------------
    # (5) 모델 선택 및 저장 기준
    # --------------------------------------------------------
    load_best_model_at_end=True,            # 학습이 끝나면 가장 성능 좋은 모델 자동 로ㅉ드
    metric_for_best_model="f1_macro",             # 최고 모델을 판단할 기준 메트릭 (f1 점수)
    greater_is_better=True,                 # 점수가 높을수록 좋은 방향 (True → f1 높을수록 좋음)

    # --------------------------------------------------------
    # (7) 로깅 / WandB / TensorBoard 보고용 설정
    # --------------------------------------------------------
    report_to=[]                            # 외부 로깅 도구(W&B, TensorBoard 등) 비활성화
)

In [50]:
# ------------------------------------------------------------
# Trainer : 모델 학습을 자동으로 관리해주는 Hugging Face 고수준 API
# ------------------------------------------------------------
trainer = Trainer(
    model=model,                   # 학습시킬 모델 (여기서는 BertClsHead)
    args=args,                     # 위에서 정의한 TrainingArguments 설정
    train_dataset=train_tok,       # 학습용 데이터셋 (Dataset 형태)
    eval_dataset=test_tok,         # 평가용 데이터셋
    tokenizer=tokenizer,           # 토크나이저 (로깅/평가 시 필요)
    compute_metrics=metrics,       # 평가 시 사용할 메트릭 함수 (accuracy, f1 등)
)

C:\Users\ekfla\AppData\Local\Temp\ipykernel_25172\2683179785.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [51]:
# ============================================================
# 7. 평가 및 예측 테스트
# ============================================================

# ------------------------------------------------------------
# (1) 평가 (validation/test 데이터로 모델 성능 확인)
# ------------------------------------------------------------
# trainer.evaluate()는 test_tok(평가용 데이터셋)을 이용해
# model.forward()를 실행하고 metrics 함수(accuracy, f1)를 계산함
eval_res = trainer.evaluate()
print("평가 결과:", eval_res)
# 출력 예시: {'eval_loss': 0.23, 'eval_accuracy': 0.91, 'eval_f1': 0.90, ...}

c:\Users\ekfla\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


평가 결과: {'eval_loss': 1.4973137378692627, 'eval_model_preparation_time': 0.0013, 'eval_accuracy': 0.23641304347826086, 'eval_f1': 0.1789987927464705, 'eval_runtime': 30.3264, 'eval_samples_per_second': 24.269, 'eval_steps_per_second': 3.034}


In [52]:
# ------------------------------------------------------------
# (2) 새 문장에 대한 예측 테스트
# ------------------------------------------------------------
# 감정분석용 샘플 문장 2개
samples = ["퇴근 후의 시간과 주말을 위해 이번에 큰마음 먹고 좋은 기능을 가진 TV를 구매해봤습니다! TV로 할 수 있는 것들이 많아지면서 제 삶이 더 즐거워졌어요~핸드폰으로만 즐기던 게임을 TV로 연동해서 커다란 화면으로 즐길 수 있다는 게 정말 신나더라고요. 게다가 손쉬운 방법으로 OTT 서비스를 이용할 수 있어서 여가 시간이 정말 기대되요!제가 영화 보는 것을 좋아해서 선명한 화질을 갖춘 TV를 구매했어요. 이전에 쓰던 TV와는 달리, 영화의 분위기를 영화관에서처럼 재현해내는 화질의 선명함에 깜짝 놀랐습니다. TV 구매를 고민하는 분들께 이 TV를 추천해 드려요!", "이거 영 불편한데요. 제품 사진이랑 좀 다른 느낌이네요"]

# ------------------------------------------------------------
# (3) 입력 문장 토크나이징 (KoBERT 입력 형식으로 변환)
# ------------------------------------------------------------
enc = tokenizer(
    samples,                        # 입력 문장 리스트
    return_tensors="pt",             # PyTorch 텐서 형태로 반환
    padding=True,                    # 배치 내 문장 길이 맞추기 (패딩 자동 추가)
    truncation=True,                  # 최대 길이 초과 시 잘라냄
    max_length=128
).to(model.classifier.weight.device)  # 모델이 올라간 디바이스(GPU/MPS/CPU)로 이동


In [53]:
# ------------------------------------------------------------
# (4) 모델 추론 (예측)
# ------------------------------------------------------------
# torch.no_grad() : 추론(inference) 시에는 gradient 계산 비활성화 → 메모리 절약, 속도 향상
with torch.no_grad():
    out = model(**enc)               # 모델에 입력 전달 (forward 수행)
    probs = torch.softmax(           # 출력 logits을 확률로 변환 (0~1 사이)
        out["logits"], dim=-1
    ).cpu().numpy()                  # GPU → CPU로 이동 후 NumPy 배열로 변환


In [54]:
# ------------------------------------------------------------
# (5) 예측 결과 출력
# ------------------------------------------------------------
# probs: 각 문장의 [부정 확률, 긍정 확률] 형태의 배열
for s, p in zip(samples, probs):
    print(f"[{s}] → 부정={p[0]:.3f},중간={p[1]:.3f},  긍정={p[2]:.3f}, 예측={p.argmax()}")
    # p.argmax() : 확률이 더 큰 쪽(0=부정, 1=긍정)을 최종 예측으로 선택


[퇴근 후의 시간과 주말을 위해 이번에 큰마음 먹고 좋은 기능을 가진 TV를 구매해봤습니다! TV로 할 수 있는 것들이 많아지면서 제 삶이 더 즐거워졌어요~핸드폰으로만 즐기던 게임을 TV로 연동해서 커다란 화면으로 즐길 수 있다는 게 정말 신나더라고요. 게다가 손쉬운 방법으로 OTT 서비스를 이용할 수 있어서 여가 시간이 정말 기대되요!제가 영화 보는 것을 좋아해서 선명한 화질을 갖춘 TV를 구매했어요. 이전에 쓰던 TV와는 달리, 영화의 분위기를 영화관에서처럼 재현해내는 화질의 선명함에 깜짝 놀랐습니다. TV 구매를 고민하는 분들께 이 TV를 추천해 드려요!] → 부정=0.410,중간=0.447,  긍정=0.143, 예측=1
[이거 영 불편한데요. 제품 사진이랑 좀 다른 느낌이네요] → 부정=0.423,중간=0.417,  긍정=0.160, 예측=0
